# Truth Archive EDA (Remote Parquet)

This notebook performs core-profile EDA by fetching data directly from Trump Truth Social Parquet archive: `https://ix.cnn.io/data/truth-social/truth_archive.parquet`.

## Setup

In [1]:
import io

import pandas as pd
import plotly.express as px
import requests

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## Load Data

In [ ]:
ARCHIVE_URL = "https://ix.cnn.io/data/truth-social/truth_archive.parquet"
HTTP_HEADERS = {"User-Agent": "ttsfeed/0.1 (Truth Social archive tracker)"}
TIMEOUT_SECONDS = 120

resp = requests.get(ARCHIVE_URL, timeout=TIMEOUT_SECONDS, headers=HTTP_HEADERS)
resp.raise_for_status()

raw_bytes = resp.content
df_raw = pd.read_parquet(io.BytesIO(raw_bytes))


print(f"Fetched {len(raw_bytes) / 1_000_000:.2f} MB from {ARCHIVE_URL}")

## Data Cleaning

In [3]:
df = df_raw.replace("", pd.NA)

In [4]:
df["created_at"] = pd.to_datetime(df["created_at"])

## Schema & Missing Values

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31625 entries, 0 to 31624
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   id                31625 non-null  object             
 1   created_at        31625 non-null  datetime64[ns, UTC]
 2   content           25983 non-null  object             
 3   url               31625 non-null  object             
 4   media             10903 non-null  object             
 5   replies_count     31625 non-null  int64              
 6   reblogs_count     31625 non-null  int64              
 7   favourites_count  31625 non-null  int64              
dtypes: datetime64[ns, UTC](1), int64(3), object(4)
memory usage: 1.9+ MB


In [6]:
missing_df = (
    df.isna()
    .agg(["sum", "mean"])
    .T
    .rename(columns={"sum": "missing_count", "mean": "missing_pct"})
    .reset_index(names="column")
)
missing_df["missing_pct"] = missing_df["missing_pct"].round(6)
missing_df = missing_df.sort_values(["missing_count", "column"], ascending=[False, True]).reset_index(drop=True)
missing_df

,column,missing_count,missing_pct
0,media,20722.0,0.655241
1,content,5642.0,0.178403
2,created_at,0.0,0.000000
3,favourites_count,0.0,0.000000
4,id,0.0,0.000000
5,reblogs_count,0.0,0.000000
6,replies_count,0.0,0.000000
7,url,0.0,0.000000


In [7]:
(df["media"].isna() & df["content"].isna()).sum()

2

In [8]:
df[df["media"].isna() & df["content"].isna()]

,id,created_at,content,url,media,replies_count,reblogs_count,favourites_count
15384,112044239738550427,2024-03-05 17:23:39.830000+00:00,<NA>,https://truthsocial.com/@realDonaldTrump/11204...,<NA>,63,480,3098
15385,112044239731983483,2024-03-05 17:23:39.730000+00:00,<NA>,https://truthsocial.com/@realDonaldTrump/11204...,<NA>,157,888,6508


## Summary Statistics

In [9]:
df.describe()

,replies_count,reblogs_count,favourites_count
count,31625.000000,31625.000000,31625.000000
mean,770.374387,3405.465138,13214.250055
std,1323.483384,3228.775028,12316.093619
min,0.000000,0.000000,0.000000
25%,120.000000,1464.000000,5629.000000
50%,396.000000,2870.000000,11199.000000
75%,961.000000,4629.000000,18054.000000
max,60886.000000,123196.000000,410837.000000


In [10]:
df["created_at"].describe(datetime_is_numeric=True)

count                                  31625
mean     2024-03-18 08:16:03.358105088+00:00
min         2022-02-14 15:54:32.528000+00:00
25%      2023-06-30 18:01:09.116999936+00:00
50%      2024-02-19 18:41:21.723000064+00:00
75%      2024-11-04 19:54:36.153999872+00:00
max         2026-02-21 04:32:29.567000+00:00
Name: created_at, dtype: object

## Temporal Analysis

In [11]:
posts_by_date = df.assign(created_date=df["created_at"].dt.date)["created_date"].value_counts().sort_index()

In [12]:
px.line(
    posts_by_date,
    labels={"index": "Date", "value": "Number of Posts"},
    title="Posts by Date",
).update_layout(showlegend=False)

In [13]:
posts_per_hour_utc = df.assign(created_hour=df["created_at"].dt.hour)["created_hour"].value_counts().sort_index()
posts_per_hour_ny = (
    df.assign(
        created_hour_ny=(
            df["created_at"]
            .dt.tz_convert("America/New_York")
            .dt.hour
        )
    )["created_hour_ny"]
    .value_counts()
    .sort_index()
)

In [14]:
px.bar(
    posts_per_hour_utc,
    labels={"index": "Hour of Day", "value": "Number of Posts"},
    title="Posts by Hour (UTC)",
).update_layout(showlegend=False)

In [15]:
px.bar(
    posts_per_hour_ny,
    labels={"index": "Hour of Day", "value": "Number of Posts"},
    title="Posts by Hour (NY)",
).update_layout(showlegend=False)

## Day-of-Week Analysis

In [16]:
posts_per_dow = (
    df.assign(
        dow=df["created_at"]
        .dt.tz_convert("America/New_York")
        .dt.day_name()
    )["dow"]
    .value_counts()
    .reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
)
px.bar(posts_per_dow, labels={"index": "Day", "value": "Posts"}, title="Posts by Day of Week (NY)").update_layout(showlegend=False)

## Engagement Distribution

In [17]:
px.histogram(df, x="replies_count")

In [18]:
px.histogram(df, x="reblogs_count")

In [19]:
px.histogram(df, x="favourites_count")

## Content Length Analysis

In [20]:
df_text = df[df["content"].notna()].copy()
df_text["word_count"] = df_text["content"].str.split().str.len()
df_text["char_count"] = df_text["content"].str.len()

px.histogram(df_text, x="word_count", nbins=60, title="Distribution of Post Word Count")

In [21]:
px.histogram(df_text, x="char_count", nbins=60, title="Distribution of Post Character Count")

## Top Posts

In [22]:
top10 = (
    df.nlargest(10, "favourites_count")
    [["created_at", "favourites_count", "reblogs_count", "replies_count", "content", "url"]]
    .assign(content=df["content"].str[:120] + "…")
    .reset_index(drop=True)
)
top10

,created_at,favourites_count,reblogs_count,replies_count,content,url
0,2022-04-28 21:29:28.207000+00:00,410837,123196,60886,I’M BACK! #COVFEFE…,https://truthsocial.com/@realDonaldTrump/10821...
1,2024-07-14 00:42:35.818000+00:00,266321,60805,58679,I want to thank The United States Secret Servi...,https://truthsocial.com/@realDonaldTrump/11278...
2,2022-02-14 15:54:32.528000+00:00,264075,49001,33994,Get Ready! Your favorite President will see y...,https://truthsocial.com/@realDonaldTrump/10779...
3,2022-04-29 22:45:26.592000+00:00,217254,47614,19641,Thank you to all of the GREAT and BEAUTIFUL Am...,https://truthsocial.com/@realDonaldTrump/10821...
4,2024-07-14 11:36:53.854000+00:00,186762,39267,22375,Thank you to everyone for your thoughts and pr...,https://truthsocial.com/@realDonaldTrump/11278...
5,2024-07-14 18:09:37.413000+00:00,157852,24717,18006,"Based on yesterday’s terrible events, I was go...",https://truthsocial.com/@realDonaldTrump/11278...
6,2025-02-28 18:16:10.692000+00:00,148322,32143,32560,A Statement from President Donald J. Trump“We ...,https://truthsocial.com/@realDonaldTrump/11408...
7,2022-08-08 22:51:32.008000+00:00,147926,55973,31871,<NA>,https://truthsocial.com/@realDonaldTrump/10878...
8,2022-05-01 00:42:59.167000+00:00,127621,28611,4682,"Truth continues to be NUMBER ONE, by a lot. Th...",https://truthsocial.com/@realDonaldTrump/10822...
9,2025-09-10 20:40:49.845000+00:00,127204,31246,18642,"The Great, and even Legendary, Charlie Kirk, i...",https://truthsocial.com/@realDonaldTrump/11518...
